In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import igraph as ig
from networkx.algorithms import bipartite
import seaborn as sns

color_map_cat = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black', 'ROE': 'orange'}
color_map_cat_no_roe = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black'}

# Hyperlink

In [ ]:
# Load the data
h_nodes = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='nodes')
h_edges = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='edges')
h_edges_clean = h_edges.copy()
# remove where the columns 'Source' and 'Target' are the same
h_edges_clean = h_edges_clean[h_edges_clean['Source'] != h_edges_clean['Target']]

# remove duplicates
h_edges_clean = h_edges_clean.drop_duplicates(subset=['Source', 'Target'], keep='first')
# CATEGORY ANALYSIS

categories_h = h_nodes.groupby('Category').size().reset_index(name='counts')
categories_h = categories_h.sort_values(by='counts', ascending=False)
categories_h = categories_h.reset_index(drop=True)
# add the counts of nodes with STATUS == "IN" in each category
categories_h['Seeds'] = categories_h['Category'].map(
    h_nodes[h_nodes['STATUS'] == 'IN'].groupby('Category').size()
)
categories_h['Seeds'] = categories_h['Seeds'].fillna(0)
categories_h['Seeds'] = categories_h['Seeds'].astype(int)
# add the counts of nodes with STATUS == "DISCOVERED" in each category
categories_h['Discovered'] = categories_h['Category'].map(
    h_nodes[h_nodes['STATUS'] == 'DISCOVERED'].groupby('Category').size()
)
categories_h['Discovered'] = categories_h['Discovered'].fillna(0)
categories_h['Discovered'] = categories_h['Discovered'].astype(int)

# add the number of Sub-categories in each category
categories_h['Sub-categories'] = categories_h['Category'].map(
    h_nodes.groupby('Category')['Sub-category'].nunique()
)
categories_h['Sub-categories'] = categories_h['Sub-categories'].fillna(0)
categories_h['Sub-categories'] = categories_h['Sub-categories'].astype(int)

edge_categories = h_edges_clean.merge(h_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# rename the column 'Category' to 'Source_Category'
edge_categories.rename(columns={'Category': 'Source_Category'}, inplace=True)
edge_categories = edge_categories.merge(h_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')
# rename the column 'Category' to 'Target_Category'
edge_categories.rename(columns={'Category': 'Target_Category'}, inplace=True)

inter_cat = edge_categories.groupby(['Source_Category', 'Target_Category']).size().reset_index(name='counts')
intra_cat = inter_cat[inter_cat['Source_Category'] == inter_cat['Target_Category']]

# Merge h_edges with h_nodes to get the category for each target node
edge_categories = h_edges_clean.merge(h_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')

# Group by Category to count in-coming edges
in_edges = edge_categories.groupby('Category').size().reset_index(name='Tot_In-strength')

# Update categories_h with the in-coming edges counts
categories_h = categories_h.drop(columns=['Tot_In-strength'], errors='ignore').merge(in_edges, on='Category', how='left')
categories_h['Tot_In-strength'] = categories_h['Tot_In-strength'].fillna(0).astype(int)

# Merge h_edges with h_nodes to get the category for each source node
edge_categories = h_edges_clean.merge(h_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')

# Group by Category to count out-going edges
out_edges = edge_categories.groupby('Category').size().reset_index(name='Tot_Out-strength')

# Update categories_h with the out-going edges counts
categories_h = categories_h.drop(columns=['Tot_Out-strength'], errors='ignore').merge(out_edges, on='Category', how='left')
categories_h['Tot_Out-strength'] = categories_h['Tot_Out-strength'].fillna(0).astype(int)

categories_h['Intra-category'] = categories_h['Category'].map(
    intra_cat.groupby('Source_Category')['counts'].sum()
)
categories_h['Intra-category'] = categories_h['Intra-category'].fillna(0)
categories_h['Intra-category'] = categories_h['Intra-category'].astype(int)

categories_h['Extra-category'] = categories_h['Tot_Out-strength'] - categories_h['Intra-category']
categories_h['Extra-category'] = categories_h['Extra-category'].fillna(0)
categories_h['Extra-category'] = categories_h['Extra-category'].astype(int)

# Seeds/Discovered strengths
# Out-strength of seeds (edges leaving seed sources)
edge_src_status = h_edges_clean.merge(h_nodes[['ID', 'Category', 'STATUS']], left_on='Source', right_on='ID', how='left')
seeds_out = edge_src_status[edge_src_status['STATUS'] == 'IN'] \
    .groupby('Category').size().reset_index(name='Seeds_Out-strength')

# In-strength of seeds (edges arriving to seed targets)
edge_tgt_status = h_edges_clean.merge(h_nodes[['ID', 'Category', 'STATUS']], left_on='Target', right_on='ID', how='left')
seeds_in = edge_tgt_status[edge_tgt_status['STATUS'] == 'IN'] \
    .groupby('Category').size().reset_index(name='Seeds_In-strength')

# In-strength of discovered nodes (edges arriving to discovered targets)
disc_in = edge_tgt_status[edge_tgt_status['STATUS'] == 'DISCOVERED'] \
    .groupby('Category').size().reset_index(name='Discovered_In-strength')

# Merge into categories_h
categories_h = categories_h.drop(columns=['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength'], errors='ignore')
categories_h = categories_h.merge(seeds_out, on='Category', how='left')
categories_h = categories_h.merge(seeds_in, on='Category', how='left')
categories_h = categories_h.merge(disc_in, on='Category', how='left')

categories_h[['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength']] = (
    categories_h[['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength']].fillna(0).astype(int)
)

In [ ]:
categories_h

In [ ]:
cols = ['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength']

perc_df = categories_h[['Category']].copy()
perc_df[cols] = (categories_h[cols].div(categories_h[cols].sum(), axis=1) * 100).round(2)
perc_df = perc_df.rename(columns={
    'Seeds_Out-strength': 'Seeds_Out_%',
    'Seeds_In-strength': 'Seeds_In_%',
    'Discovered_In-strength': 'Discovered_In_%'
})

perc_df

In [ ]:
# three side-by-side donut charts for Seeds_Out-strength, Seeds_In-strength, and Discovered_In-strength
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

labels = labels if 'labels' in globals() else categories_h['Category'].tolist()
colors = colors if 'colors' in globals() else [color_map_cat.get(cat, 'gray') for cat in labels]
metrics = [
    ('Seeds_Out-strength', "Seeds\n Out-Degree"),
    ('Seeds_In-strength', "Seeds\n In-Degree"),
    ('Discovered_In-strength', "Discovered\n In-Degree"),
]

for ax, (col, title) in zip(axes, metrics):
    vals = categories_h[col].tolist()
    data = [(lab, v, c) for lab, v, c in zip(labels, vals, colors) if v > 0]

    if not data:
        ax.axis('off')
        ax.set_title(title, fontsize=20)
        continue

    flabels, fvals, fcolors = map(list, zip(*data))
    wedges, texts, autotexts = ax.pie(
        fvals,
        labels=flabels,
        autopct=lambda p: f'{p:.1f}%' if p > 0 else '',
        startangle=0,
        colors=fcolors,
        wedgeprops={'alpha': 0.6},
        textprops={'fontsize': 24}
    )
    centre = plt.Circle((0, 0), 0.30, fc='white')
    ax.add_artist(centre)
    ax.axis('equal')
    ax.text(0, 0, title, ha='center', va='center', fontsize=18, fontweight='bold')
    #ax.set_title(title, fontsize=20)

plt.tight_layout()
plt.show()



In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 8))

labels = categories_h['Category'].tolist()
values = categories_h['Tot_In-strength'].tolist()
colors = [color_map_cat.get(cat, 'gray') for cat in labels]

wedges1, texts1, autotexts1 = ax1.pie(
    values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=0,
    colors=colors,
    wedgeprops={'alpha': 0.6},
    textprops={'fontsize': 18}
)

centre1 = plt.Circle((0, 0), 0.30, fc='white')
ax1.add_artist(centre1)
ax1.axis('equal')

plt.text(0, 0, 'Hyperlink', horizontalalignment='center', verticalalignment='center', fontsize=22, fontweight='bold')

plt.tight_layout()
plt.show()



In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 8))

labels = categories_h['Category'].tolist()
values = categories_h['Tot_In-strength'].tolist()
colors = [color_map_cat.get(cat, 'gray') for cat in labels]

wedges1, texts1, autotexts1 = ax1.pie(
    values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=0,
    colors=colors,
    wedgeprops={'alpha': 0.6},
    textprops={'fontsize': 18}
)

centre1 = plt.Circle((0, 0), 0.30, fc='white')
ax1.add_artist(centre1)
ax1.axis('equal')

plt.text(0, 0, 'Hyperlink', horizontalalignment='center', verticalalignment='center', fontsize=22, fontweight='bold')

plt.tight_layout()
plt.show()



In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 8))

labels = categories_h['Category'].tolist()
values = categories_h['Discovered'].tolist()
colors = [color_map_cat.get(cat, 'gray') for cat in labels]

wedges1, texts1, autotexts1 = ax1.pie(
    values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=0,
    colors=colors,
    wedgeprops={'alpha': 0.6},
    textprops={'fontsize': 18}
)

centre1 = plt.Circle((0, 0), 0.30, fc='white')
ax1.add_artist(centre1)
ax1.axis('equal')

plt.text(0, 0, 'Hyperlink', horizontalalignment='center', verticalalignment='center', fontsize=22, fontweight='bold')

plt.tight_layout()
plt.show()



# Twitter

In [ ]:
# Use a common order and colors for both charts
order = labels if 'labels' in globals() else categories_h['Category'].tolist()
plot_colors = colors if 'colors' in globals() else [color_map_cat.get(cat, 'gray') for cat in order]

h_map = categories_h.set_index('Category')['Discovered'].to_dict()
tw_map = categories_tw.set_index('Category')['Discovered'].to_dict()
vals_h = [h_map.get(cat, 0) for cat in order]
vals_tw = [tw_map.get(cat, 0) for cat in order]

for vals, title in [(vals_h, 'Hyperlink'), (vals_tw, 'Twitter')]:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.pie(
        vals,
        labels=order,
        autopct='%1.1f%%',
        startangle=0,
        colors=plot_colors,
        wedgeprops={'alpha': 0.6},
        textprops={'fontsize': 18}
    )
    ax.add_artist(plt.Circle((0, 0), 0.30, fc='white'))
    ax.axis('equal')
    plt.text(0, 0, title, ha='center', va='center', fontsize=22, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
# Load the data
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')
# Add Status
tw_nodes = tw_nodes.merge(h_nodes[['ID', 'STATUS']], left_on='ID', right_on='ID', how='left')
tw_edges_clean = tw_edges.copy()
# remove where the columns 'Source' and 'Target' are the same
tw_edges_clean = tw_edges_clean[tw_edges_clean['Source'] != tw_edges_clean['Target']]
# remove duplicates
#tw_edges_clean = tw_edges_clean.drop_duplicates(subset=['Source', 'Target'], keep='first')
weighted_tw_edges = tw_edges_clean.copy()
# count the number of edges between each pair of nodes
weighted_tw_edges = weighted_tw_edges.groupby(['Source', 'Target']).size().reset_index(name='weight')
#weighted_tw_edges

# CATEGORY ANALYSIS

categories_tw = tw_nodes.groupby('Category').size().reset_index(name='counts')
categories_tw = categories_tw.sort_values(by='counts', ascending=False)
categories_tw = categories_tw.reset_index(drop=True)
# add the counts of nodes with STATUS == "IN" in each category
categories_tw['Seeds'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'IN'].groupby('Category').size()
)

categories_tw['Seeds'] = categories_tw['Seeds'].fillna(0)
categories_tw['Seeds'] = categories_tw['Seeds'].astype(int)
# add the counts of nodes with STATUS == "DISCOVERED" in each category
categories_tw['Discovered'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'DISCOVERED'].groupby('Category').size()
)
categories_tw['Discovered'] = categories_tw['Discovered'].fillna(0)
categories_tw['Discovered'] = categories_tw['Discovered'].astype(int)
# add the number of Sub-categories in each category
categories_tw['Sub-categories'] = categories_tw['Category'].map(
    tw_nodes.groupby('Category')['Sub-category'].nunique()
)
categories_tw['Sub-categories'] = categories_tw['Sub-categories'].fillna(0)
categories_tw['Sub-categories'] = categories_tw['Sub-categories'].astype(int)


edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# rename the column 'Category' to 'Source_Category'
edge_categories_tw.rename(columns={'Category': 'Source_Category'}, inplace=True)

edge_categories_tw = edge_categories_tw.merge(tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')
# rename the column 'Category' to 'Target_Category'
edge_categories_tw.rename(columns={'Category': 'Target_Category'}, inplace=True)

inter_cat_tw = edge_categories_tw.groupby(['Source_Category', 'Target_Category'])['weight'].sum().reset_index(name='counts')
intra_cat_tw = inter_cat_tw[inter_cat_tw['Source_Category'] == inter_cat_tw['Target_Category']]

# Merge h_edges with h_nodes to get the category for each target node
edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')

# Group by Category to count in-coming edges
in_edges_tw = edge_categories_tw.groupby('Category')['weight'].sum().reset_index(name='Tot_In-strength')
# Update categories_tw with the in-coming edges counts
categories_tw = categories_tw.drop(columns=['Tot_In-strength'], errors='ignore').merge(in_edges_tw, on='Category', how='left')
categories_tw['Tot_In-strength'] = categories_tw['Tot_In-strength'].fillna(0).astype(int)
# Merge h_edges with h_nodes to get the category for each source node
edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# Group by Category to count out-going edges
out_edges_tw = edge_categories_tw.groupby('Category')['weight'].sum().reset_index(name='Tot_Out-strength')
# Update categories_tw with the out-going edges counts
categories_tw = categories_tw.drop(columns=['Tot_Out-strength'], errors='ignore').merge(out_edges_tw, on='Category', how='left')
categories_tw['Tot_Out-strength'] = categories_tw['Tot_Out-strength'].fillna(0).astype(int)
categories_tw['Intra-category'] = categories_tw['Category'].map(
    intra_cat_tw.groupby('Source_Category')['counts'].sum()
)
categories_tw['Intra-category'] = categories_tw['Intra-category'].fillna(0)
categories_tw['Intra-category'] = categories_tw['Intra-category'].astype(int)
categories_tw['Extra-category'] = categories_tw['Tot_Out-strength'] - categories_tw['Intra-category']

categories_tw['Extra-category'] = categories_tw['Extra-category'].fillna(0)
categories_tw['Extra-category'] = categories_tw['Extra-category'].astype(int)
# Seeds/Discovered strengths
# Out-strength of seeds (edges leaving seed sources)
edge_src_status_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category', 'STATUS']], left_on='Source', right_on='ID', how='left')
seeds_out_tw = edge_src_status_tw[edge_src_status_tw['STATUS'] == 'IN'] \
    .groupby('Category')['weight'].sum().reset_index(name='Seeds_Out-strength')
# In-strength of seeds (edges arriving to seed targets)
edge_tgt_status_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category', 'STATUS']], left_on='Target', right_on='ID', how='left')
seeds_in_tw = edge_tgt_status_tw[edge_tgt_status_tw['STATUS'] == 'IN'] \
    .groupby('Category')['weight'].sum().reset_index(name='Seeds_In-strength')
# In-strength of discovered nodes (edges arriving to discovered targets)
disc_in_tw = edge_tgt_status_tw[edge_tgt_status_tw['STATUS'] == 'DISCOVERED'] \
    .groupby('Category')['weight'].sum().reset_index(name='Discovered_In-strength')
# Merge into categories_tw
categories_tw = categories_tw.drop(columns=['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength'], errors='ignore')
categories_tw = categories_tw.merge(seeds_out_tw, on='Category', how='left')
categories_tw = categories_tw.merge(seeds_in_tw, on='Category', how='left')
categories_tw = categories_tw.merge(disc_in_tw, on='Category', how='left')
categories_tw[['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength']] = (
    categories_tw[['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength']].fillna(0).astype(int)
)
categories_tw['Discovered_Percentage'] = (categories_tw['Seeds_In-strength'] / categories_tw['Seeds_In-strength'].sum()) * 100
categories_tw['Discovered_Percentage'] = categories_tw['Discovered_Percentage'].round(2)
categories_tw[['Category', 'Discovered_Percentage']]

In [ ]:
# Unweighted analysis: treat each unique Source–Target as one edge

# 1) Build unweighted edge list (drop self-loops and duplicates)
tw_edges_clean = tw_edges[tw_edges['Source'] != tw_edges['Target']].drop_duplicates(subset=['Source', 'Target'])
weighted_tw_edges = tw_edges_clean[['Source', 'Target']].copy()
weighted_tw_edges['weight'] = 1  # constant 1 per unique edge

# 2) Category-level node stats
categories_tw = (
    tw_nodes.groupby('Category').size().reset_index(name='counts')
    .sort_values(by='counts', ascending=False).reset_index(drop=True)
)
categories_tw['Seeds'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'IN'].groupby('Category').size()
).fillna(0).astype(int)
categories_tw['Discovered'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'DISCOVERED'].groupby('Category').size()
).fillna(0).astype(int)
categories_tw['Sub-categories'] = categories_tw['Category'].map(
    tw_nodes.groupby('Category')['Sub-category'].nunique()
).fillna(0).astype(int)

# 3) Inter/Intra-category edge counts (unweighted)
edge_categories_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left'
).rename(columns={'Category': 'Source_Category'})
edge_categories_tw = edge_categories_tw.merge(
    tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left'
).rename(columns={'Category': 'Target_Category'})
inter_cat_tw = edge_categories_tw.groupby(
    ['Source_Category', 'Target_Category']
)['weight'].sum().reset_index(name='counts')
intra_cat_tw = inter_cat_tw[inter_cat_tw['Source_Category'] == inter_cat_tw['Target_Category']]

# 4) In/Out edge counts per category (unweighted)
edge_categories_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left'
)
in_edges_tw = edge_categories_tw.groupby('Category')['weight'].sum().reset_index(name='Tot_In-strength')
categories_tw = categories_tw.drop(columns=['Tot_In-strength'], errors='ignore').merge(in_edges_tw, on='Category', how='left')
categories_tw['Tot_In-strength'] = categories_tw['Tot_In-strength'].fillna(0).astype(int)

edge_categories_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left'
)
out_edges_tw = edge_categories_tw.groupby('Category')['weight'].sum().reset_index(name='Tot_Out-strength')
categories_tw = categories_tw.drop(columns=['Tot_Out-strength'], errors='ignore').merge(out_edges_tw, on='Category', how='left')
categories_tw['Tot_Out-strength'] = categories_tw['Tot_Out-strength'].fillna(0).astype(int)

categories_tw['Intra-category'] = categories_tw['Category'].map(
    intra_cat_tw.groupby('Source_Category')['counts'].sum()
).fillna(0).astype(int)
categories_tw['Extra-category'] = (categories_tw['Tot_Out-strength'] - categories_tw['Intra-category']).fillna(0).astype(int)

# 5) Seeds/Discovered edge counts (unweighted)
edge_src_status_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category', 'STATUS']], left_on='Source', right_on='ID', how='left'
)
seeds_out_tw = edge_src_status_tw[edge_src_status_tw['STATUS'] == 'IN'].groupby('Category')['weight'].sum().reset_index(name='Seeds_Out-strength')

edge_tgt_status_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category', 'STATUS']], left_on='Target', right_on='ID', how='left'
)
seeds_in_tw = edge_tgt_status_tw[edge_tgt_status_tw['STATUS'] == 'IN'].groupby('Category')['weight'].sum().reset_index(name='Seeds_In-strength')
disc_in_tw = edge_tgt_status_tw[edge_tgt_status_tw['STATUS'] == 'DISCOVERED'].groupby('Category')['weight'].sum().reset_index(name='Discovered_In-strength')

categories_tw = categories_tw.drop(columns=['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength'], errors='ignore')
categories_tw = categories_tw.merge(seeds_out_tw, on='Category', how='left')
categories_tw = categories_tw.merge(seeds_in_tw, on='Category', how='left')
categories_tw = categories_tw.merge(disc_in_tw, on='Category', how='left')
categories_tw[['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength']] = (
    categories_tw[['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength']].fillna(0).astype(int)
)

# 6) Share metric (same formula, now on unweighted counts)
total_seeds_out = categories_tw['Seeds_Out-strength'].sum()
categories_tw['Discovered_Percentage'] = (categories_tw['Seeds_Out-strength'] / total_seeds_out * 100).fillna(0).round(2)

categories_tw[['Category', 'Discovered_Percentage']]




In [ ]:
# Unweighted analysis: treat each unique Source–Target as one edge

# 1) Build unweighted edge list (drop self-loops and duplicates)
tw_edges_clean = tw_edges[tw_edges['Source'] != tw_edges['Target']].drop_duplicates(subset=['Source', 'Target'])
weighted_tw_edges = tw_edges_clean[['Source', 'Target']].copy()
weighted_tw_edges['weight'] = 1  # constant 1 per unique edge

# 2) Category-level node stats
categories_tw = (
    tw_nodes.groupby('Category').size().reset_index(name='counts')
    .sort_values(by='counts', ascending=False).reset_index(drop=True)
)
categories_tw['Seeds'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'IN'].groupby('Category').size()
).fillna(0).astype(int)
categories_tw['Discovered'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'DISCOVERED'].groupby('Category').size()
).fillna(0).astype(int)
categories_tw['Sub-categories'] = categories_tw['Category'].map(
    tw_nodes.groupby('Category')['Sub-category'].nunique()
).fillna(0).astype(int)

# 3) Inter/Intra-category edge counts (unweighted)
edge_categories_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left'
).rename(columns={'Category': 'Source_Category'})
edge_categories_tw = edge_categories_tw.merge(
    tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left'
).rename(columns={'Category': 'Target_Category'})
inter_cat_tw = edge_categories_tw.groupby(
    ['Source_Category', 'Target_Category']
)['weight'].sum().reset_index(name='counts')
intra_cat_tw = inter_cat_tw[inter_cat_tw['Source_Category'] == inter_cat_tw['Target_Category']]

# 4) In/Out edge counts per category (unweighted)
edge_categories_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left'
)
in_edges_tw = edge_categories_tw.groupby('Category')['weight'].sum().reset_index(name='Tot_In-strength')
categories_tw = categories_tw.drop(columns=['Tot_In-strength'], errors='ignore').merge(in_edges_tw, on='Category', how='left')
categories_tw['Tot_In-strength'] = categories_tw['Tot_In-strength'].fillna(0).astype(int)

edge_categories_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left'
)
out_edges_tw = edge_categories_tw.groupby('Category')['weight'].sum().reset_index(name='Tot_Out-strength')
categories_tw = categories_tw.drop(columns=['Tot_Out-strength'], errors='ignore').merge(out_edges_tw, on='Category', how='left')
categories_tw['Tot_Out-strength'] = categories_tw['Tot_Out-strength'].fillna(0).astype(int)

categories_tw['Intra-category'] = categories_tw['Category'].map(
    intra_cat_tw.groupby('Source_Category')['counts'].sum()
).fillna(0).astype(int)
categories_tw['Extra-category'] = (categories_tw['Tot_Out-strength'] - categories_tw['Intra-category']).fillna(0).astype(int)

# 5) Seeds/Discovered edge counts (unweighted)
edge_src_status_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category', 'STATUS']], left_on='Source', right_on='ID', how='left'
)
seeds_out_tw = edge_src_status_tw[edge_src_status_tw['STATUS'] == 'IN'].groupby('Category')['weight'].sum().reset_index(name='Seeds_Out-strength')

edge_tgt_status_tw = weighted_tw_edges.merge(
    tw_nodes[['ID', 'Category', 'STATUS']], left_on='Target', right_on='ID', how='left'
)
seeds_in_tw = edge_tgt_status_tw[edge_tgt_status_tw['STATUS'] == 'IN'].groupby('Category')['weight'].sum().reset_index(name='Seeds_In-strength')
disc_in_tw = edge_tgt_status_tw[edge_tgt_status_tw['STATUS'] == 'DISCOVERED'].groupby('Category')['weight'].sum().reset_index(name='Discovered_In-strength')

categories_tw = categories_tw.drop(columns=['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength'], errors='ignore')
categories_tw = categories_tw.merge(seeds_out_tw, on='Category', how='left')
categories_tw = categories_tw.merge(seeds_in_tw, on='Category', how='left')
categories_tw = categories_tw.merge(disc_in_tw, on='Category', how='left')
categories_tw[['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength']] = (
    categories_tw[['Seeds_Out-strength', 'Seeds_In-strength', 'Discovered_In-strength']].fillna(0).astype(int)
)

# 6) Share metric (same formula, now on unweighted counts)
total_seeds_out = categories_tw['Discovered_In-strength'].sum()
categories_tw['Discovered_Percentage'] = (categories_tw['Discovered_In-strength'] / total_seeds_out * 100).fillna(0).round(2)

categories_tw[['Category', 'Discovered_Percentage']]




In [ ]:
categories_tw

In [ ]:
# three side-by-side donut charts for Seeds_Out-strength, Seeds_In-strength, and Discovered_In-strength
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

labels = labels if 'labels' in globals() else categories_tw['Category'].tolist()
colors = colors if 'colors' in globals() else [color_map_cat.get(cat, 'gray') for cat in labels]
metrics = [
    ('Seeds_Out-strength', "Seeds\n Out-Degree"),
    ('Seeds_In-strength', "Seeds\n In-Degree"),
    ('Discovered_In-strength', "Discovered\n In-Degree"),
]

for ax, (col, title) in zip(axes, metrics):
    vals = categories_tw[col].tolist()
    data = [(lab, v, c) for lab, v, c in zip(labels, vals, colors) if v > 0]

    if not data:
        ax.axis('off')
        ax.set_title(title, fontsize=20)
        continue

    flabels, fvals, fcolors = map(list, zip(*data))
    wedges, texts, autotexts = ax.pie(
        fvals,
        labels=flabels,
        autopct=lambda p: f'{p:.1f}%' if p > 0 else '',
        startangle=0,
        colors=fcolors,
        wedgeprops={'alpha': 0.6},
        textprops={'fontsize': 24}
    )
    centre = plt.Circle((0, 0), 0.30, fc='white')
    ax.add_artist(centre)
    ax.axis('equal')
    ax.text(0, 0, title, ha='center', va='center', fontsize=18, fontweight='bold')
    #ax.set_title(title, fontsize=20)

plt.tight_layout()
plt.show()



In [ ]:
# three side-by-side donut charts for Seeds_Out-strength, Seeds_In-strength, and Discovered_In-strength
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

labels = labels if 'labels' in globals() else categories_tw['Category'].tolist()
colors = colors if 'colors' in globals() else [color_map_cat.get(cat, 'gray') for cat in labels]
metrics = [
    ('Seeds_Out-strength', "Seeds\n Out-Degree"),
    ('Seeds_In-strength', "Seeds\n In-Degree"),
    ('Discovered_In-strength', "Discovered\n In-Degree"),
]

for ax, (col, title) in zip(axes, metrics):
    vals = categories_tw[col].tolist()
    data = [(lab, v, c) for lab, v, c in zip(labels, vals, colors) if v > 0]

    if not data:
        ax.axis('off')
        ax.set_title(title, fontsize=20)
        continue

    flabels, fvals, fcolors = map(list, zip(*data))
    wedges, texts, autotexts = ax.pie(
        fvals,
        labels=flabels,
        autopct=lambda p: f'{p:.1f}%' if p > 0 else '',
        startangle=0,
        colors=fcolors,
        wedgeprops={'alpha': 0.6},
        textprops={'fontsize': 24}
    )
    centre = plt.Circle((0, 0), 0.30, fc='white')
    ax.add_artist(centre)
    ax.axis('equal')
    ax.text(0, 0, title, ha='center', va='center', fontsize=18, fontweight='bold')
    #ax.set_title(title, fontsize=20)

plt.tight_layout()
plt.show()



In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 8))

# use Twitter categories instead of hyperlink categories
labels = categories_tw['Category'].tolist()
values = categories_tw['Discovered'].tolist()
colors = [color_map_cat.get(cat, 'gray') for cat in labels]

wedges1, texts1, autotexts1 = ax1.pie(
    values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=0,
    colors=colors,
    wedgeprops={'alpha': 0.6},
    textprops={'fontsize': 18}
)

centre1 = plt.Circle((0, 0), 0.30, fc='white')
ax1.add_artist(centre1)
ax1.axis('equal')

plt.text(0, 0, 'Twitter', horizontalalignment='center', verticalalignment='center', fontsize=22, fontweight='bold')

plt.tight_layout()
plt.show()



In [ ]:
categories_h

In [ ]:
# order categories_h by the Category column and put 'ROE' last
categories_h = categories_h.sort_values(by='Category')
roe_h = categories_h[categories_h['Category'] == 'ROE']
categories_h = categories_h[categories_h['Category'] != 'ROE']
categories_h = pd.concat([categories_h, roe_h], ignore_index=True)

# order categories_tw by the Category column and put 'ROE' last
categories_tw = categories_tw.sort_values(by='Category')
roe_tw = categories_tw[categories_tw['Category'] == 'ROE']
categories_tw = categories_tw[categories_tw['Category'] != 'ROE']
categories_tw = pd.concat([categories_tw, roe_tw], ignore_index=True)

In [ ]:
color_map_cat = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black', 'ROE': 'orange'}
color_map_cat_no_roe = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black'}

In [ ]:
categories_tw

In [ ]:
color_map_cat

In [ ]:
# categories_h to a latex table
categories_h_latex = categories_h[['Category', 'counts', 'Sub-categories', 'Seeds', 'Discovered', 'Tot_In-strength', 'Tot_Out-strength', 'Intra-category', 'Extra-category']]
categories_h_latex = categories_h_latex.rename(columns={'Category': 'Category', 'counts': 'Counts', 'Sub-categories': 'Sub-categories', 'Seeds': 'Seeds', 'Discovered': 'Discovered', 'Tot_In-strength': 'Tot In-strength', 'Tot_Out-strength': 'Tot Out-strength', 'Intra-category': 'Intra-category', 'Extra-category': 'Extra-category'})
categories_h_latex = categories_h_latex.set_index('Category')
categories_h_latex = categories_h_latex.rename_axis(index=None)
categories_h_latex = categories_h_latex.astype(str)
categories_h_latex = categories_h_latex.replace({'0': '0', '1': '1', '2': '2', '3': '3', '4': '4', '5': '5', '6': '6', '7': '7', '8': '8', '9': '9'})
categories_h_latex = categories_h_latex.replace({'0.0': '0', '1.0': '1', '2.0': '2', '3.0': '3', '4.0': '4', '5.0': '5', '6.0': '6', '7.0': '7', '8.0': '8', '9.0': '9'})
categories_h_latex = categories_h_latex.replace({'nan': '0'})


# print the latex table
print(categories_h_latex.to_latex(escape=False, index=True, column_format='lccccccccc', float_format='%.0f', multirow=True, multicolumn=True, label='tab:categories_hyperlink', caption='Counts of nodes in each category and their connections.'))

In [ ]:
edge_categories = h_edges_clean.merge(h_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# rename the column 'Category' to 'Source_Category'
edge_categories.rename(columns={'Category': 'Source_Category'}, inplace=True)

edge_categories = edge_categories.merge(h_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')
# rename the column 'Category' to 'Target_Category'
edge_categories.rename(columns={'Category': 'Target_Category'}, inplace=True)

inter_cat = edge_categories.groupby(['Source_Category', 'Target_Category']).size().reset_index(name='counts')

#inter_cat['Tot_Out-strength'] = inter_cat['Source_Category'].map(
#    edge_categories.groupby('Source_Category').size()
#)

inter_cat

In [ ]:
# add counts divided by Tot_Out-strength
inter_cat['weight_source'] = inter_cat['counts'] / inter_cat['Source_Category'].map(
    categories_h.set_index('Category')['Tot_Out-strength']
)
inter_cat['weight_source'] = inter_cat['weight_source'].fillna(0)
inter_cat['weight_source'] = inter_cat['weight_source'].astype(float)
inter_cat

In [ ]:
categories_h

In [ ]:
# add counts divided by Tot_Out-strength
inter_cat['weight_target'] = inter_cat['counts'] / inter_cat['Target_Category'].map(
    categories_h.set_index('Category')['Tot_In-strength']
)
inter_cat['weight_target'] = inter_cat['weight_target'].fillna(0)
inter_cat['weight_target'] = inter_cat['weight_target'].astype(float)
inter_cat

In [ ]:
inter_cat['Tot_Out-strength'] = inter_cat['Source_Category'].map(
    edge_categories.groupby('Source_Category').size()
)
inter_cat['Tot_Out-strength'] = inter_cat['Tot_Out-strength'].fillna(0)
inter_cat['Tot_Out-strength'] = inter_cat['Tot_Out-strength'].astype(int)

In [ ]:
inter_cat['Tot_In-strength'] = inter_cat['Target_Category'].map(
    edge_categories.groupby('Target_Category').size()
)
inter_cat['Tot_In-strength'] = inter_cat['Tot_In-strength'].fillna(0)
inter_cat['Tot_In-strength'] = inter_cat['Tot_In-strength'].astype(int)

In [ ]:
inter_cat

In [ ]:


matrix 

In [ ]:
matrix = inter_cat.pivot(index='Source_Category', columns='Target_Category', values='weight_source').fillna(0)
# add row 'ROE' with all zeros
matrix.loc['ROE'] = 0

# heatmap of the inter-category connections
plt.figure(figsize=(14, 10))
ax = sns.heatmap(
    matrix,
    cmap='YlGnBu',
    annot=True,
    fmt='.3f',
    annot_kws={'fontsize': 22, 'fontweight': 'bold'},
    linewidths=0.7,
    linecolor='black'
)

plt.title(
    'Number of links between categories divided by the number of links \n from the source category in the Hyperlink Network',
    fontsize=22
)
plt.xlabel('Target Category', fontsize=18)
ax.xaxis.tick_top()
plt.ylabel('Source Category', fontsize=18)
plt.xticks(rotation=45, fontsize=16)
plt.yticks(rotation=0, fontsize=16)

# enlarge colorbar tick labels
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=16)

plt.tight_layout()
#plt.savefig('inter_category_connections.png', dpi=300)
plt.show()

In [ ]:
matrix = inter_cat.pivot(index='Source_Category', columns='Target_Category', values='weight_target').fillna(0)
# add row 'ROE' with all zeros
matrix.loc['ROE'] = 0

# heatmap of the inter-category connections
plt.figure(figsize=(14, 10))
ax = sns.heatmap(
    matrix,
    cmap='YlGnBu',
    annot=True,
    fmt='.3f',
    annot_kws={'fontsize': 22, 'fontweight': 'bold'},
    linewidths=0.7,
    linecolor='black'
)

plt.title(
    'Number of links between categories divided by the number of links \n to the target category in the Hyperlink Network',
    fontsize=22
)
plt.xlabel('Target Category', fontsize=18)
ax.xaxis.tick_top()
plt.ylabel('Source Category', fontsize=18)
plt.xticks(rotation=45, fontsize=16)
plt.yticks(rotation=0, fontsize=16)

# enlarge colorbar tick labels
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=16)

plt.tight_layout()
#plt.savefig('inter_category_connections.png', dpi=300)
plt.show()

In [ ]:
# create a weighted directed network between the categories, with the weights taken from the inter_cat dataframe
G = nx.from_pandas_edgelist(inter_cat, source='Source_Category', target='Target_Category', edge_attr='counts', create_using=nx.DiGraph())

# add the number of nodes in each category as a node attribute  
for node in G.nodes():
    G.nodes[node]['counts'] = categories_h[categories_h['Category'] == node]['counts'].values[0]
    G.nodes[node]['Sub-categories'] = categories_h[categories_h['Category'] == node]['Sub-categories'].values[0]
    G.nodes[node]['Seeds'] = categories_h[categories_h['Category'] == node]['Seeds'].values[0]
    G.nodes[node]['Discovered'] = categories_h[categories_h['Category'] == node]['Discovered'].values[0]
    G.nodes[node]['Tot_In-strength'] = categories_h[categories_h['Category'] == node]['Tot_In-strength'].values[0]
    G.nodes[node]['Tot_Out-strength'] = categories_h[categories_h['Category'] == node]['Tot_Out-strength'].values[0]
    G.nodes[node]['Intra-category'] = categories_h[categories_h['Category'] == node]['Intra-category'].values[0]
    G.nodes[node]['Extra-category'] = categories_h[categories_h['Category'] == node]['Extra-category'].values[0]

# remove self-loops
G.remove_edges_from(list(nx.selfloop_edges(G)))


In [ ]:
color_map_cat = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black', 'ROE': 'orange'}
color_map_cat_no_roe = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black'}

In [ ]:
from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

# plot the network with edge width based on the counts of the edges
plt.figure(figsize=(12, 8))

# reuse existing layout/sizes/colors if available
pos = pos if 'pos' in globals() else nx.spring_layout(G, k=0.5, iterations=6)
node_sizes = node_sizes if 'node_sizes' in globals() else [(G.nodes[n]['Seeds'] + 1) * 1000 for n in G.nodes()]
node_colors = node_colors if 'node_colors' in globals() else [color_map_cat.get(n, 'gray') for n in G.nodes()]

nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.8,
    edgecolors='black',   # visible outline around nodes
    linewidths=2.0
)

edge_widths = [max(1.5, G[u][v]['counts'] / 4) for u, v in G.edges()]
arrows = nx.draw_networkx_edges(
    G, pos,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,
    connectionstyle='arc3,rad=0.12',
    min_source_margin=30,   # keep arrows off node centers
    min_target_margin=33
)

# add a white outline to arrows for better definition
try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

# no labels on nodes
# nx.draw_networkx_labels(G, pos, font_size=20, font_weight='bold')

# legend with node names
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(n, 'gray'),
           markeredgecolor='black', label=n)
    for n in G.nodes()
]
plt.legend(handles=legend_elements, title='Category',
           loc='center left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.axis('off')
plt.tight_layout()
# plt.savefig('inter_category_network.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# remove node labels
# nx.draw_networkx_labels(G_tw, pos, font_size=20, font_weight='bold')

# legend with colored dots
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=12,
           markerfacecolor=col, markeredgecolor='white', label=cat)
    for cat, col in color_map_cat.items()
]
plt.legend(handles=legend_elements, title='Category',
           loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

plt.axis('off')
plt.tight_layout()
#plt.savefig('inter_category_network.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib import patheffects as pe

# plot the network with edge width based on the counts of the edges
plt.figure(figsize=(12, 8))

# reuse existing layout if available, otherwise compute it
pos = nx.spring_layout(G, k=0.5, iterations=6)

# reuse existing node_sizes/colors if present, otherwise compute
node_sizes = node_sizes if 'node_sizes' in globals() else [(G.nodes[n]['Seeds'] + 1) * 2000 for n in G.nodes()]
node_colors = node_colors if 'node_colors' in globals() else [color_map_cat.get(n, 'gray') for n in G.nodes()]

nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.5,
    edgecolors='white',
    linewidths=1.0
)

edge_widths = [max(1.5, G[u][v]['counts'] / 4) for u, v in G.edges()]
arrows = nx.draw_networkx_edges(
    G, pos,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,               # larger arrowheads
    connectionstyle='arc3,rad=0.08'  # slight curve to reduce overlap
)

# add a white outline to arrows for better definition
try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    # fallback if a single collection is returned
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

nx.draw_networkx_labels(G, pos, font_size=20, font_weight='bold')

plt.axis('off')
plt.tight_layout()
#plt.savefig('inter_category_network.png', dpi=300)
plt.show()

In [ ]:
# plot the network with edge width based on the counts of the edges
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=0.5, iterations=50)
node_sizes = [(G.nodes[node]['Seeds']+1) * 500 for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='lightblue', alpha=0.8)
nx.draw_networkx_edges(G, pos, width=[G[u][v]['counts'] / 8 for u, v in G.edges()], alpha=0.5)
nx.draw_networkx_labels(G, pos, font_size=10)
plt.title('Directed network between categories in the Hyperlink Network')
plt.axis('off')
plt.tight_layout()
#plt.savefig('inter_category_network.png', dpi=300)
plt.show()

In [ ]:
# plot the network
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, k=0.5)
node_sizes = [(G.nodes[node]['Seeds']+1) * 500 for node in G.nodes()]
nx.draw(G, pos, with_labels=True, node_size=node_sizes, node_color='lightblue', font_size=10, font_weight='bold', arrows=True)
edge_labels = nx.get_edge_attributes(G, 'weight')
edge_labels = {k: round(v, 2) for k, v in edge_labels.items()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')
plt.title('Hyperlink Network between Categories')
plt.show()

## Sub-categories

In [ ]:
sub_categories_h = h_nodes.groupby('Sub-category').size().reset_index(name='counts')
sub_categories_h = sub_categories_h.sort_values(by='counts', ascending=False)
sub_categories_h = sub_categories_h.reset_index(drop=True)

sub_categories_h['Category'] = sub_categories_h['Sub-category'].map(
    h_nodes.groupby('Sub-category')['Category'].first()
)

# add the counts of nodes with STATUS == "IN" in each sub_category
sub_categories_h['Seeds'] = sub_categories_h['Sub-category'].map(
    h_nodes[h_nodes['STATUS'] == 'IN'].groupby('Sub-category').size()
)
sub_categories_h['Seeds'] = sub_categories_h['Seeds'].fillna(0)
sub_categories_h['Seeds'] = sub_categories_h['Seeds'].astype(int)
# add the counts of nodes with STATUS == "DISCOVERED" in each sub_category
sub_categories_h['Discovered'] = sub_categories_h['Sub-category'].map(
    h_nodes[h_nodes['STATUS'] == 'DISCOVERED'].groupby('Sub-category').size()
)
sub_categories_h['Discovered'] = sub_categories_h['Discovered'].fillna(0)
sub_categories_h['Discovered'] = sub_categories_h['Discovered'].astype(int)


edge_sub_categories = h_edges.merge(h_nodes[['ID', 'Sub-category']], left_on='Source', right_on='ID', how='left')
# rename the column 'Sub-category' to 'Source_Sub-category'
edge_sub_categories.rename(columns={'Sub-category': 'Source_Sub-category'}, inplace=True)
edge_sub_categories = edge_sub_categories.merge(h_nodes[['ID', 'Sub-category']], left_on='Target', right_on='ID', how='left')
# rename the column 'Sub-category' to 'Target_Sub-category'
edge_sub_categories.rename(columns={'Sub-category': 'Target_Sub-category'}, inplace=True)
inter_sub_cat = edge_sub_categories.groupby(['Source_Sub-category', 'Target_Sub-category']).size().reset_index(name='counts')
intra_sub_cat = inter_sub_cat[inter_sub_cat['Source_Sub-category'] == inter_sub_cat['Target_Sub-category']]
# Merge h_edges with h_nodes to get the sub-category for each target node
edge_sub_categories = h_edges.merge(h_nodes[['ID', 'Sub-category']], left_on='Target', right_on='ID', how='left')
# Group by Sub-category to count in-coming edges
in_edges_sub = edge_sub_categories.groupby('Sub-category').size().reset_index(name='Tot_In-strength')
# Update sub_categories_h with the in-coming edges counts
sub_categories_h = sub_categories_h.drop(columns=['Tot_In-strength'], errors='ignore').merge(in_edges_sub, on='Sub-category', how='left')
sub_categories_h['Tot_In-strength'] = sub_categories_h['Tot_In-strength'].fillna(0).astype(int)
# Merge h_edges with h_nodes to get the sub-category for each source node
edge_sub_categories = h_edges.merge(h_nodes[['ID', 'Sub-category']], left_on='Source', right_on='ID', how='left')
# Group by Sub-category to count out-going edges
out_edges_sub = edge_sub_categories.groupby('Sub-category').size().reset_index(name='Tot_Out-strength')
# Update sub_categories_h with the out-going edges counts
sub_categories_h = sub_categories_h.drop(columns=['Tot_Out-strength'], errors='ignore').merge(out_edges_sub, on='Sub-category', how='left')
sub_categories_h['Tot_Out-strength'] = sub_categories_h['Tot_Out-strength'].fillna(0).astype(int)
sub_categories_h['Intra-category'] = sub_categories_h['Sub-category'].map(
    intra_sub_cat.groupby('Source_Sub-category')['counts'].sum()
)
sub_categories_h['Intra-category'] = sub_categories_h['Intra-category'].fillna(0)
sub_categories_h['Intra-category'] = sub_categories_h['Intra-category'].astype(int)
sub_categories_h['Extra-category'] = sub_categories_h['Tot_Out-strength'] - sub_categories_h['Intra-category']
sub_categories_h['Extra-category'] = sub_categories_h['Extra-category'].fillna(0)
sub_categories_h['Extra-category'] = sub_categories_h['Extra-category'].astype(int)



In [ ]:
sub_categories_h

# Twitter

In [ ]:
# Load the data
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')
# Add Status
tw_nodes = tw_nodes.merge(h_nodes[['ID', 'STATUS']], left_on='ID', right_on='ID', how='left')

In [ ]:
tw_edges_clean = tw_edges.copy()
# remove where the columns 'Source' and 'Target' are the same
tw_edges_clean = tw_edges_clean[tw_edges_clean['Source'] != tw_edges_clean['Target']]
# remove duplicates
#tw_edges_clean = tw_edges_clean.drop_duplicates(subset=['Source', 'Target'], keep='first')

In [ ]:
weighted_tw_edges = tw_edges_clean.copy()
# count the number of edges between each pair of nodes
weighted_tw_edges = weighted_tw_edges.groupby(['Source', 'Target']).size().reset_index(name='weight')
weighted_tw_edges


## Categories

In [ ]:
# weighted

categories_tw = tw_nodes.groupby('Category').size().reset_index(name='counts')
categories_tw = categories_tw.sort_values(by='counts', ascending=False)
categories_tw = categories_tw.reset_index(drop=True)
# add the counts of nodes with STATUS == "IN" in each category
categories_tw['Seeds'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'IN'].groupby('Category').size()
)

categories_tw['Seeds'] = categories_tw['Seeds'].fillna(0)
categories_tw['Seeds'] = categories_tw['Seeds'].astype(int)
# add the counts of nodes with STATUS == "DISCOVERED" in each category
categories_tw['Discovered'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'DISCOVERED'].groupby('Category').size()
)
categories_tw['Discovered'] = categories_tw['Discovered'].fillna(0)
categories_tw['Discovered'] = categories_tw['Discovered'].astype(int)
# add the number of Sub-categories in each category
categories_tw['Sub-categories'] = categories_tw['Category'].map(
    tw_nodes.groupby('Category')['Sub-category'].nunique()
)
categories_tw['Sub-categories'] = categories_tw['Sub-categories'].fillna(0)
categories_tw['Sub-categories'] = categories_tw['Sub-categories'].astype(int)


edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# rename the column 'Category' to 'Source_Category'
edge_categories_tw.rename(columns={'Category': 'Source_Category'}, inplace=True)

edge_categories_tw = edge_categories_tw.merge(tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')
# rename the column 'Category' to 'Target_Category'
edge_categories_tw.rename(columns={'Category': 'Target_Category'}, inplace=True)

inter_cat_tw = edge_categories_tw.groupby(['Source_Category', 'Target_Category'])['weight'].sum().reset_index(name='counts')
intra_cat_tw = inter_cat_tw[inter_cat_tw['Source_Category'] == inter_cat_tw['Target_Category']]

# Merge h_edges with h_nodes to get the category for each target node
edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')

# Group by Category to count in-coming edges
in_edges_tw = edge_categories_tw.groupby('Category')['weight'].sum().reset_index(name='Tot_In-strength')
# Update categories_tw with the in-coming edges counts
categories_tw = categories_tw.drop(columns=['Tot_In-strength'], errors='ignore').merge(in_edges_tw, on='Category', how='left')
categories_tw['Tot_In-strength'] = categories_tw['Tot_In-strength'].fillna(0).astype(int)
# Merge h_edges with h_nodes to get the category for each source node
edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# Group by Category to count out-going edges
out_edges_tw = edge_categories_tw.groupby('Category')['weight'].sum().reset_index(name='Tot_Out-strength')
# Update categories_tw with the out-going edges counts
categories_tw = categories_tw.drop(columns=['Tot_Out-strength'], errors='ignore').merge(out_edges_tw, on='Category', how='left')
categories_tw['Tot_Out-strength'] = categories_tw['Tot_Out-strength'].fillna(0).astype(int)
categories_tw['Intra-category'] = categories_tw['Category'].map(
    intra_cat_tw.groupby('Source_Category')['counts'].sum()
)
categories_tw['Intra-category'] = categories_tw['Intra-category'].fillna(0)
categories_tw['Intra-category'] = categories_tw['Intra-category'].astype(int)
categories_tw['Extra-category'] = categories_tw['Tot_Out-strength'] - categories_tw['Intra-category']

In [ ]:
categories_tw

In [ ]:
categories_tw

In [ ]:
# binary

categories_tw = tw_nodes.groupby('Category').size().reset_index(name='counts')
categories_tw = categories_tw.sort_values(by='counts', ascending=False)
categories_tw = categories_tw.reset_index(drop=True)
# add the counts of nodes with STATUS == "IN" in each category
categories_tw['Seeds'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'IN'].groupby('Category').size()
)

categories_tw['Seeds'] = categories_tw['Seeds'].fillna(0)
categories_tw['Seeds'] = categories_tw['Seeds'].astype(int)
# add the counts of nodes with STATUS == "DISCOVERED" in each category
categories_tw['Discovered'] = categories_tw['Category'].map(
    tw_nodes[tw_nodes['STATUS'] == 'DISCOVERED'].groupby('Category').size()
)
categories_tw['Discovered'] = categories_tw['Discovered'].fillna(0)
categories_tw['Discovered'] = categories_tw['Discovered'].astype(int)
# add the number of Sub-categories in each category
categories_tw['Sub-categories'] = categories_tw['Category'].map(
    tw_nodes.groupby('Category')['Sub-category'].nunique()
)
categories_tw['Sub-categories'] = categories_tw['Sub-categories'].fillna(0)
categories_tw['Sub-categories'] = categories_tw['Sub-categories'].astype(int)


edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# rename the column 'Category' to 'Source_Category'
edge_categories_tw.rename(columns={'Category': 'Source_Category'}, inplace=True)

edge_categories_tw = edge_categories_tw.merge(tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')
# rename the column 'Category' to 'Target_Category'
edge_categories_tw.rename(columns={'Category': 'Target_Category'}, inplace=True)

inter_cat_tw = edge_categories_tw.groupby(['Source_Category', 'Target_Category']).size().reset_index(name='counts')
intra_cat_tw = inter_cat_tw[inter_cat_tw['Source_Category'] == inter_cat_tw['Target_Category']]

# Merge h_edges with h_nodes to get the category for each target node
edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')
# Group by Category to count in-coming edges
in_edges_tw = edge_categories_tw.groupby('Category').size().reset_index(name='Tot_In-strength')
# Update categories_tw with the in-coming edges counts
categories_tw = categories_tw.drop(columns=['Tot_In-strength'], errors='ignore').merge(in_edges_tw, on='Category', how='left')
categories_tw['Tot_In-strength'] = categories_tw['Tot_In-strength'].fillna(0).astype(int)
# Merge h_edges with h_nodes to get the category for each source node
edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# Group by Category to count out-going edges
out_edges_tw = edge_categories_tw.groupby('Category').size().reset_index(name='Tot_Out-strength')
# Update categories_tw with the out-going edges counts
categories_tw = categories_tw.drop(columns=['Tot_Out-strength'], errors='ignore').merge(out_edges_tw, on='Category', how='left')
categories_tw['Tot_Out-strength'] = categories_tw['Tot_Out-strength'].fillna(0).astype(int)
categories_tw['Intra-category'] = categories_tw['Category'].map(
    intra_cat_tw.groupby('Source_Category')['counts'].sum()
)
categories_tw['Intra-category'] = categories_tw['Intra-category'].fillna(0)
categories_tw['Intra-category'] = categories_tw['Intra-category'].astype(int)
categories_tw['Extra-category'] = categories_tw['Tot_Out-strength'] - categories_tw['Intra-category']

In [ ]:
categories_tw

In [ ]:
# categories_tw to a latex table
categories_tw_latex = categories_tw[['Category', 'counts', 'Sub-categories', 'Seeds', 'Discovered', 'Tot_In-strength', 'Tot_Out-strength', 'Intra-category', 'Extra-category']]
categories_tw_latex = categories_tw_latex.rename(columns={'Category': 'Category', 'counts': 'Counts', 'Sub-categories': 'Sub-cat.', 'Seeds': 'Seeds', 'Discovered': 'Discovered', 'Tot_In-strength': 'Tot In', 'Tot_Out-strength': 'Tot Out', 'Intra-category': 'Intra', 'Extra-category': 'Extra'})
categories_tw_latex = categories_tw_latex.set_index('Category')
categories_tw_latex = categories_tw_latex.rename_axis(index=None)
categories_tw_latex = categories_tw_latex.astype(str)
categories_tw_latex = categories_tw_latex.replace({'0': '0', '1': '1', '2': '2', '3': '3', '4': '4', '5': '5', '6': '6', '7': '7', '8': '8', '9': '9'})
categories_tw_latex = categories_tw_latex.replace({'0.0': '0', '1.0': '1', '2.0': '2', '3.0': '3', '4.0': '4', '5.0': '5', '6.0': '6', '7.0': '7', '8.0': '8', '9.0': '9'})
categories_tw_latex = categories_tw_latex.replace({'nan': '0'})
# print the latex table
print(categories_tw_latex.to_latex(escape=False, index=True, column_format='lccccccccc', float_format='%.0f', multirow=True, multicolumn=True, label='tab:categories_twitter', caption='Counts of nodes in each category and their connections.'))

In [ ]:
edge_categories_tw

In [ ]:
weighted_tw_edges

In [ ]:
edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# rename the column 'Category' to 'Source_Category'
edge_categories_tw.rename(columns={'Category': 'Source_Category'}, inplace=True)

edge_categories_tw = edge_categories_tw.merge(tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')
# rename the column 'Category' to 'Target_Category'
edge_categories_tw.rename(columns={'Category': 'Target_Category'}, inplace=True)

inter_cat_tw = edge_categories_tw.groupby(['Source_Category', 'Target_Category']).size().reset_index(name='counts')

inter_cat_tw

In [ ]:
edge_categories_tw = weighted_tw_edges.merge(tw_nodes[['ID', 'Category']], left_on='Source', right_on='ID', how='left')
# rename the column 'Category' to 'Source_Category'
edge_categories_tw.rename(columns={'Category': 'Source_Category'}, inplace=True)

edge_categories_tw = edge_categories_tw.merge(tw_nodes[['ID', 'Category']], left_on='Target', right_on='ID', how='left')
# rename the column 'Category' to 'Target_Category'
edge_categories_tw.rename(columns={'Category': 'Target_Category'}, inplace=True)

inter_cat_tw = edge_categories_tw.groupby(['Source_Category', 'Target_Category'], as_index=False)['weight'].sum().rename(columns={'weight': 'counts'})

inter_cat_tw

In [ ]:
# add counts divided by Tot_Out-strength
inter_cat_tw['weight'] = inter_cat_tw['counts'] / inter_cat_tw['Source_Category'].map(
    categories_tw.set_index('Category')['Tot_Out-strength']
)
inter_cat_tw['weight'] = inter_cat_tw['weight'].fillna(0)
inter_cat_tw['weight'] = inter_cat_tw['weight'].astype(float)

inter_cat_tw['Tot_Out-strength'] = inter_cat_tw['Source_Category'].map(
    edge_categories_tw.groupby('Source_Category').size()
)
inter_cat_tw['Tot_Out-strength'] = inter_cat_tw['Tot_Out-strength'].fillna(0)
inter_cat_tw['Tot_Out-strength'] = inter_cat_tw['Tot_Out-strength'].astype(int)

matrix_tw = inter_cat_tw.pivot(index='Source_Category', columns='Target_Category', values='weight').fillna(0)
# add row 'ROE' with all zeros
matrix_tw.loc['ROE'] = 0
matrix_tw.loc['OTHER'] = 0

matrix_tw

In [ ]:
# heatmap of the inter-category connections
plt.figure(figsize=(14, 10))
ax = sns.heatmap(
    matrix_tw,
    cmap='YlGnBu',
    annot=True,
    fmt='.3f',
    annot_kws={'fontsize': 22, 'fontweight': 'bold'},
    linewidths=0.7,
    linecolor='black'
)
plt.title('Number of links between categories divided by the number of links \n from the source category in the Twitter Weighted Network', fontsize=22)
plt.xlabel('Target Category', fontsize=18)
ax.xaxis.tick_top()
plt.ylabel('Source Category', fontsize=18)
plt.xticks(rotation=45, fontsize=16)
plt.yticks(rotation=0, fontsize=16)
# enlarge colorbar tick labels
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=16)
plt.tight_layout()
#plt.savefig('inter_category_connections_tw.png', dpi=300)
plt.show()

In [ ]:
inter_cat_tw['weight_in'] = inter_cat_tw['counts'] / inter_cat_tw['Target_Category'].map(
    categories_tw.set_index('Category')['Tot_In-strength']
)
inter_cat_tw['weight_in'] = inter_cat_tw['weight_in'].fillna(0)
inter_cat_tw['weight_in'] = inter_cat_tw['weight_in'].astype(float)

matrix_tw_in = inter_cat_tw.pivot(index='Source_Category', columns='Target_Category', values='weight_in').fillna(0)
# add row 'ROE' with all zeros
matrix_tw_in.loc['ROE'] = 0
matrix_tw_in.loc['OTHER'] = 0

# heatmap of the inter-category connections
plt.figure(figsize=(14, 10))
ax = sns.heatmap(matrix_tw_in,
                 cmap='YlGnBu', annot=True, fmt='.3f',
                 annot_kws={'fontsize': 22, 'fontweight': 'bold'},
                 linewidths=0.7, linecolor='black')
plt.title('Number of links between categories divided by the number of links \n to the target category in the Twitter Weighted Network', fontsize=22)
plt.xlabel('Target Category', fontsize=18)
ax.xaxis.tick_top()
plt.ylabel('Source Category', fontsize=18)
plt.xticks(rotation=45, fontsize=16)
plt.yticks(rotation=0, fontsize=16)
# enlarge colorbar tick labels
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=16)
plt.tight_layout()
#plt.savefig('inter_category_connections_tw_in.png', dpi=300)
plt.show()

In [ ]:
# heatmap of the inter-category connections
plt.figure(figsize=(14, 10))
ax = sns.heatmap(
    matrix_tw,
    cmap='YlGnBu',
    annot=True,
    fmt='.3f',
    annot_kws={'fontsize': 22, 'fontweight': 'bold'},
    linewidths=0.7,
    linecolor='black'
)
plt.title('Number of links between categories divided by the number of links \n from the source category in the Twitter Binary Network', fontsize=22)
plt.xlabel('Target Category', fontsize=18)
ax.xaxis.tick_top()
plt.ylabel('Source Category', fontsize=18)
plt.xticks(rotation=45, fontsize=16)
plt.yticks(rotation=0, fontsize=16)
# enlarge colorbar tick labels
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=16)
plt.tight_layout()
#plt.savefig('inter_category_connections_tw.png', dpi=300)
plt.show()

In [ ]:
inter_cat_tw['weight_in'] = inter_cat_tw['counts'] / inter_cat_tw['Target_Category'].map(
    categories_tw.set_index('Category')['Tot_In-strength']
)
inter_cat_tw['weight_in'] = inter_cat_tw['weight_in'].fillna(0)
inter_cat_tw['weight_in'] = inter_cat_tw['weight_in'].astype(float)

matrix_tw_in = inter_cat_tw.pivot(index='Source_Category', columns='Target_Category', values='weight_in').fillna(0)
# add row 'ROE' with all zeros
matrix_tw_in.loc['ROE'] = 0
matrix_tw_in.loc['OTHER'] = 0

# heatmap of the inter-category connections
plt.figure(figsize=(14, 10))
ax = sns.heatmap(matrix_tw_in,
                 cmap='YlGnBu', annot=True, fmt='.3f',
                 annot_kws={'fontsize': 22, 'fontweight': 'bold'},
                 linewidths=0.7, linecolor='black')
plt.title('Number of links between categories divided by the number of links \n to the target category in the Twitter Binary Network', fontsize=22)
plt.xlabel('Target Category', fontsize=18)
ax.xaxis.tick_top()
plt.ylabel('Source Category', fontsize=18)
plt.xticks(rotation=45, fontsize=16)
plt.yticks(rotation=0, fontsize=16)
# enlarge colorbar tick labels
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=16)
plt.tight_layout()
#plt.savefig('inter_category_connections_tw_in.png', dpi=300)
plt.show()

In [ ]:
inter_cat_tw

In [ ]:
# create a weighted directed network between the categories, with the weights taken from the inter_cat dataframe
G_tw = nx.from_pandas_edgelist(inter_cat_tw, source='Source_Category', target='Target_Category', edge_attr='counts', create_using=nx.DiGraph())
# add the number of nodes in each category as a node attribute

for node in G_tw.nodes():
    G_tw.nodes[node]['counts'] = categories_tw[categories_tw['Category'] == node]['counts'].values[0]
    G_tw.nodes[node]['Sub-categories'] = categories_tw[categories_tw['Category'] == node]['Sub-categories'].values[0]
    G_tw.nodes[node]['Seeds'] = categories_tw[categories_tw['Category'] == node]['Seeds'].values[0]
    G_tw.nodes[node]['Discovered'] = categories_tw[categories_tw['Category'] == node]['Discovered'].values[0]
    G_tw.nodes[node]['Tot_In-strength'] = categories_tw[categories_tw['Category'] == node]['Tot_In-strength'].values[0]
    G_tw.nodes[node]['Tot_Out-strength'] = categories_tw[categories_tw['Category'] == node]['Tot_Out-strength'].values[0]
    G_tw.nodes[node]['Intra-category'] = categories_tw[categories_tw['Category'] == node]['Intra-category'].values[0]
    G_tw.nodes[node]['Extra-category'] = categories_tw[categories_tw['Category'] == node]['Extra-category'].values[0]

    # remove self-loops
G_tw.remove_edges_from(list(nx.selfloop_edges(G_tw)))

In [ ]:
from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

# plot the network with edge width based on the counts of the edges
plt.figure(figsize=(12, 8))

# reuse existing layout/sizes/colors if available
pos = pos if 'pos' in globals() else nx.spring_layout(G_tw, k=0.5, iterations=6)
node_sizes = node_sizes if 'node_sizes' in globals() else [(G_tw.nodes[n]['Seeds'] + 1) * 1000 for n in G_tw.nodes()]
node_colors = node_colors if 'node_colors' in globals() else [color_map_cat.get(n, 'gray') for n in G_tw.nodes()]

nx.draw_networkx_nodes(
    G_tw, pos,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.8,
    edgecolors='black',   # visible outline around nodes
    linewidths=2.0
)

edge_widths = [max(1.5, G_tw[u][v]['counts'] / 6) for u, v in G_tw.edges()]
arrows = nx.draw_networkx_edges(
    G_tw, pos,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,
    connectionstyle='arc3,rad=0.12',
    min_source_margin=30,   # keep arrows off node centers
    min_target_margin=33
)

# add a white outline to arrows for better definition
try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

# no labels on nodes
# nx.draw_networkx_labels(G, pos, font_size=20, font_weight='bold')

# legend with node names
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(n, 'gray'),
           markeredgecolor='black', label=n)
    for n in G.nodes()
]
plt.legend(handles=legend_elements, title='Category',
           loc='center left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Twitter Binary Network', fontsize=22)

plt.axis('off')
plt.tight_layout()
# plt.savefig('inter_category_network.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

# plot the network with edge width based on the counts of the edges
plt.figure(figsize=(12, 8))

# reuse existing layout/sizes/colors if available
pos = pos if 'pos' in globals() else nx.spring_layout(G_tw, k=0.5, iterations=6)
node_sizes = node_sizes if 'node_sizes' in globals() else [(G_tw.nodes[n]['Seeds'] + 1) * 1000 for n in G_tw.nodes()]
node_colors = node_colors if 'node_colors' in globals() else [color_map_cat.get(n, 'gray') for n in G_tw.nodes()]

nx.draw_networkx_nodes(
    G_tw, pos,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.8,
    edgecolors='black',   # visible outline around nodes
    linewidths=2.0
)

edge_widths = [max(1.5, G_tw[u][v]['counts'] / 40) for u, v in G_tw.edges()]
arrows = nx.draw_networkx_edges(
    G_tw, pos,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,
    connectionstyle='arc3,rad=0.12',
    min_source_margin=30,   # keep arrows off node centers
    min_target_margin=33
)

# add a white outline to arrows for better definition
try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

# no labels on nodes
# nx.draw_networkx_labels(G, pos, font_size=20, font_weight='bold')

# legend with node names
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(n, 'gray'),
           markeredgecolor='black', label=n)
    for n in G.nodes()
]

plt.title('Twitter Weighted Network', fontsize=22)

plt.axis('off')
plt.tight_layout()
# plt.savefig('inter_category_network.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib import patheffects as pe

# plot the network with edge width based on the counts of the edges
plt.figure(figsize=(12, 8))

# reuse existing layout if available, otherwise compute it
pos = nx.spring_layout(G, k=0.5, iterations=6)

# reuse existing node_sizes/colors if present, otherwise compute
node_sizes = node_sizes if 'node_sizes' in globals() else [(G_tw.nodes[n]['Seeds'] + 1) * 2000 for n in G_tw.nodes()]
node_colors = node_colors if 'node_colors' in globals() else [color_map_cat.get(n, 'gray') for n in G_tw.nodes()]

nx.draw_networkx_nodes(
    G_tw, pos,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.5,
    edgecolors='white',
    linewidths=1.0
)

edge_widths = [max(1.5, G_tw[u][v]['weight'] / 4) for u, v in G_tw.edges()]
arrows = nx.draw_networkx_edges(
    G_tw, pos,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,               # larger arrowheads
    connectionstyle='arc3,rad=0.08'  # slight curve to reduce overlap
)

# add a white outline to arrows for better definition
try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    # fallback if a single collection is returned
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

nx.draw_networkx_labels(G_tw, pos, font_size=20, font_weight='bold')

plt.axis('off')
plt.tight_layout()
#plt.savefig('inter_category_network.png', dpi=300)
plt.show()

In [ ]:
# pallini senza nomi, ma con legenda

In [ ]:
from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

# plot the network with edge width based on the counts of the edges
plt.figure(figsize=(12, 8))

# reuse existing layout if available, otherwise compute it
pos = pos or nx.spring_layout(G_tw, k=0.5, iterations=6)

# reuse existing node_sizes/colors if present, otherwise compute
node_sizes =  [(G_tw.nodes[n]['Seeds'] + 1) * 300 for n in G_tw.nodes()]
node_colors =  [color_map_cat.get(n, 'gray') for n in G_tw.nodes()]

nx.draw_networkx_nodes(
    G_tw, pos,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.87,
    edgecolors='white',
    linewidths=1.0
)

edge_widths = [max(1.5, G_tw[u][v]['weight'] / .003) for u, v in G_tw.edges()]
arrows = nx.draw_networkx_edges(
    G_tw, pos,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=20,               # larger arrowheads
    connectionstyle='arc3,rad=0.12'  # slight curve to reduce overlap
)

# add a white outline to arrows for better definition
try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    # fallback if a single collection is returned
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

# remove node labels
# nx.draw_networkx_labels(G_tw, pos, font_size=20, font_weight='bold')

# legend with colored dots
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=12,
           markerfacecolor=col, markeredgecolor='white', label=cat)
    for cat, col in color_map_cat.items()
]
plt.legend(handles=legend_elements, title='Category',
           loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

plt.axis('off')
plt.tight_layout()
#plt.savefig('inter_category_network.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot the network with edge width based on the counts of the edges
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G_tw, k=0.5, iterations=50)
node_sizes = [(G_tw.nodes[node]['Seeds']+1) * 500 for node in G_tw.nodes()]
nx.draw_networkx_nodes(G_tw, pos, node_size=node_sizes, node_color='lightblue', alpha=0.8)
nx.draw_networkx_edges(G_tw, pos, width=[G_tw[u][v]['weight'] * 15 for u, v in G_tw.edges()], alpha=0.5)
nx.draw_networkx_labels(G_tw, pos, font_size=10)
plt.title('Directed network between categories in the Twitter Network')
plt.axis('off')
plt.tight_layout()
#plt.savefig('inter_category_network_tw.png', dpi=300)
plt.show()

In [ ]:
# plot the network
plt.figure(figsize=(8, 6))
pos_tw = nx.spring_layout(G_tw, k=0.5)
node_sizes_tw = [(G_tw.nodes[node]['Seeds']+1) * 500 for node in G_tw.nodes()]
nx.draw(G_tw, pos_tw, with_labels=True, node_size=node_sizes_tw, node_color='lightblue', font_size=10, font_weight='bold', arrows=True) 
edge_labels_tw = nx.get_edge_attributes(G_tw, 'weight')
edge_labels_tw = {k: round(v, 2) for k, v in edge_labels_tw.items()}
nx.draw_networkx_edge_labels(G_tw, pos_tw, edge_labels=edge_labels_tw, font_color='red')
plt.title('Twitter Network between Categories')
plt.show()
